# 01. YOLO11n 가중치 받기와 크기 재기

**목표**
- 사전학습된 가중치(파라미터)를 내려받는다
- 파라미터 수와 파일 용량을 직접 확인한다

**용어**
- **파라미터 = 가중치(weight)**: 모델이 학습하면서 맞춰 놓은 숫자들. CNN에서는 주로 필터(커널) 값입니다.
- **사전학습(pretrained)**: 다른 사람이 큰 데이터셋(COCO, 80개 클래스)으로 미리 학습해 둔 가중치.
- **`.pt` 파일**: 그 숫자들을 저장한 PyTorch 파일.

In [1]:
import torch
import ultralytics

print("ultralytics:", ultralytics.__version__)
print("torch      :", torch.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())

ultralytics: 8.4.143
torch      : 2.14.0+cu130
GPU 사용 가능: True


## 1. 가중치 받기
`YOLO("yolo11n.pt")` 를 실행하면, 이 폴더에 파일이 없을 때 **자동으로 다운로드**합니다 (약 5MB).

In [2]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

In [4]:
from pathlib import Path

size_mb = Path("yolo11n.pt").stat().st_size / 1024**2
print(f"파일 용량: {size_mb:.1f} MiB")

파일 용량: 5.4 MiB


## 2. 파라미터 수 확인

### 방법 A: Ultralytics 요약
`model.info()` 는 층(layer) 수, 파라미터 수, GFLOPs(계산량)를 한 줄로 보여 줍니다.

In [5]:
model.info()

YOLO11n summary: 181 layers, 2,624,080 parameters, 0 gradients, 6.7 GFLOPs


(181, 2624080, 0, 6.6733184)

In [6]:
model

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, bias=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=

### 방법 B: 직접 세기
모델은 결국 PyTorch 텐서(숫자 배열)의 묶음입니다. 텐서마다 원소 개수(`numel()`)를 더하면 전체 파라미터 수가 됩니다.

In [7]:
total = sum(p.numel() for p in model.model.parameters())
print(f"전체 파라미터: {total:,} 개 (약 {total / 1e6:.1f}M)")

# 숫자 1개를 float32 로 저장하면 4바이트, float16 이면 2바이트
print(f"float32 로 저장하면: {total * 4 / 1024**2:.1f} MB")
print(f"float16 로 저장하면: {total * 2 / 1024**2:.1f} MB")

전체 파라미터: 2,624,080 개 (약 2.6M)
float32 로 저장하면: 10.0 MB
float16 로 저장하면: 5.0 MB


In [19]:
next(model.model.parameters()).dtype

torch.float32

### 어떤 층에 파라미터가 있을까?
앞쪽 10개만 이름과 모양(shape)을 출력해 봅니다.
합성곱(Conv) 가중치의 shape 은 `[출력 채널, 입력 채널, 커널 높이, 커널 너비]` 입니다.

In [8]:
for i, (name, p) in enumerate(model.model.named_parameters()):
    if i >= 10:
        break
    print(f"{name:40s} {str(list(p.shape)):20s} {p.numel():>8,}")

model.0.conv.weight                      [16, 3, 3, 3]             432
model.0.bn.weight                        [16]                       16
model.0.bn.bias                          [16]                       16
model.1.conv.weight                      [32, 16, 3, 3]          4,608
model.1.bn.weight                        [32]                       32
model.1.bn.bias                          [32]                       32
model.2.cv1.conv.weight                  [32, 32, 1, 1]          1,024
model.2.cv1.bn.weight                    [32]                       32
model.2.cv1.bn.bias                      [32]                       32
model.2.cv2.conv.weight                  [64, 48, 1, 1]          3,072


## 3. 이 모델이 아는 클래스
사전학습 모델은 COCO 데이터셋의 80개 클래스를 압니다. 쓰레기와 관련 있어 보이는 게 있는지 찾아보세요.

In [9]:
print(len(model.names), "개 클래스")
print(model.names)

80 개 클래스
{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68

In [10]:
# 쓰레기와 관련 있어 보이는 클래스만 골라 보기 (목록은 자유롭게 고쳐 보세요)
trash_like = ["bottle", "cup", "wine glass", "fork", "knife", "spoon", "bowl", "banana", "apple"]

for idx, name in model.names.items():
    if name in trash_like:
        print(idx, name)

39 bottle
40 wine glass
41 cup
42 fork
43 knife
44 spoon
45 bowl
46 banana
47 apple


## 직접 해보기
`yolo11s.pt` (한 단계 큰 모델)도 받아서 아래 표를 채워 보세요.

| 모델 | 파라미터 | 파일 용량 | GFLOPs |
|---|---|---|---|
| yolo11n | 2.6M | 5.4MiB | 6.7GFLOPs |
| yolo11s | 9.4M | 18.42Mib | 21.8GFLOPs |

In [11]:
model_s = YOLO("yolo11s.pt")
model_s.info()

YOLO11s summary: 181 layers, 9,458,752 parameters, 0 gradients, 21.8 GFLOPs


(181, 9458752, 0, 21.8363392)

In [17]:
print(f"파일 용량: {Path("yolo11s.pt").stat().st_size / (1024**2):.2f}MiB")

파일 용량: 18.42MiB


In [18]:
next(model.model.parameters()).dtype

torch.float32

## 정리 질문
1. "파라미터 수 × 4바이트(float32)" 와 실제 `.pt` 파일 용량이 다르다. 왜 그럴까? (힌트: float16)
2. 캔(can), 비닐, 종이는 80개 클래스에 있나? 없다면 어떻게 해야 찾을 수 있을까?
3. 파라미터가 많으면 좋은 점과 나쁜 점은? (Jetson 에서 돌린다고 생각해 보기)

**내 답:**
-